# Signed quantum sawtooth map
Native FlagQuantum CPU workflow preserving all original sizes, times and 1,000 draws each. Historical IonQ data is explicitly identified.


## Native composable quantum program


In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
from IPython.display import Image, display

from examples.synthesis.applications.physical_systems.quantum_chaos.quantum_sawtooth_map.overview.workflow import (
    TIMESTEPS,
    historical_observations,
    localization_parameters,
    plot_comparison,
    run_workflow,
    verify,
)
from flagquantum.drawer import draw

## Original localization scale


In [ ]:
NUM_QUBITS = 3
KICK_MAG = 0.1
kick_bound, localization_length = localization_parameters()
assert kick_bound > KICK_MAG and localization_length < 2**NUM_QUBITS
print("Kick bound:", kick_bound, "localization length:", localization_length)

## Synthesize the unchanged signed program


In [ ]:
one_step = verify(1)
print(draw(one_step.program.circuit))

## Count the actual exact target sequence


In [ ]:
depth = one_step.resources.depth
cx_count = one_step.resources.operation_counts["cx"]
print("Actual exact CX/U3 one-step depth:", depth, "CX count:", cx_count)
print("Absolute native/target state errors:", one_step.maximum_native_error, one_step.maximum_target_error)

## Original four execution times


In [ ]:
timesteps = TIMESTEPS
print("Original timesteps:", timesteps)

## Distinguish extrapolation from actual resources


In [ ]:
estimated_depths = tuple(depth * t for t in timesteps)
estimated_cxs = tuple(cx_count * t for t in timesteps)
print("Source-style one-step depth extrapolation (estimate):", estimated_depths)
print("Source-style one-step CX extrapolation (estimate):", estimated_cxs)
print("Total estimated CX count:", sum(estimated_cxs))

## Original shot count


In [ ]:
NUM_SHOTS = 1000
SEED = 1970

## Execute and sample all original times


In [ ]:
rows = run_workflow(shots=NUM_SHOTS, seed=SEED)
for row in rows:
    resource = row.evolution.resources
    print("T=", row.evolution.kicks, "actual full depth:", resource.depth, "CX:", resource.operation_counts["cx"])
    print("Actual samples:", row.observations)

## Keep hardware disabled


In [ ]:
RUN_ON_HARDWARE = False
BACKEND_NAME = "qpu.forte-1"
print("Hardware submission disabled as in source; fresh execution is local CPU only.")

## Record no fresh hardware job


In [ ]:
hardware_status = "not_submitted"
assert not RUN_ON_HARDWARE
print("Fresh FlagQuantum hardware job:", hardware_status)

## Load verified historical measurements


In [ ]:
historical = {t: historical_observations(t) for t in timesteps}
print("Historical upstream IonQ observations; not fresh FlagQuantum hardware execution.")
print("Historical shots per timestep:", {t: sum(r.count for r in data) for t, data in historical.items()})

## Preserve the source display convention


In [ ]:
display_grid = tuple(range(-4, 5))
assert display_grid[-1] == 4
print("Signed QNum states are -4..3; +4 is a zero-filled display point.")

## Compare the observed distributions


In [ ]:
figure = plot_comparison(rows)
buffer = BytesIO()
figure.savefig(buffer, format="png")
display(Image(buffer.getvalue()))
plt.close(figure)


## Check original localization condition


In [ ]:
for row in rows:
    observed_localization = next(r.probability for r in row.observations if r.momentum == -2)
    assert observed_localization > 0.85
    assert row.exact_probabilities[2] > 0.85
    print("T=", row.evolution.kicks, "local observed P(-2)=", observed_localization)
print("Every original timestep preserves localization; historical hardware values are separate observations.")